# 02 - Calidad, limpieza y preparación

Cada decisión se documenta con evidencia, acción e impacto. La base original se preserva en `data/raw/`; todas las transformaciones se aplican sobre una copia de trabajo.


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
raw_path = ROOT / "data" / "raw" / "streaming_users_dirty.json"
processed_path = ROOT / "data" / "processed" / "streaming_users_processed.csv"
log_path = ROOT / "logs" / "pipeline_log.csv"

def normalizar_texto(valor):
    """Limpia espacios y pasa texto a minúsculas para mapear variantes."""
    if pd.isna(valor):
        return np.nan
    return str(valor).strip().lower()

def parsear_fechas_login(serie):
    """Intenta leer fechas en formatos frecuentes sin guardar columnas auxiliares."""
    parsed = pd.to_datetime(serie, errors="coerce", format="%Y-%m-%d")
    faltan = parsed.isna() & serie.notna()
    parsed.loc[faltan] = pd.to_datetime(serie.loc[faltan], errors="coerce", dayfirst=True)
    faltan = parsed.isna() & serie.notna()
    parsed.loc[faltan] = pd.to_datetime(serie.loc[faltan], errors="coerce")
    return parsed

def ordenar_usuarios_repetidos(df):
    """Ordena filas duplicadas por calidad sin agregar columnas al dataset final."""
    login_tmp = parsear_fechas_login(df["last_login_date"])
    login_valido_tmp = login_tmp.notna() & (login_tmp <= pd.Timestamp("2026-06-28"))
    consumo_valido_tmp = df["monthly_watch_time_mins"].between(0, 14400, inclusive="both")
    consumo_tipico = df.loc[consumo_valido_tmp, "monthly_watch_time_mins"].median()
    distancia_consumo = (df["monthly_watch_time_mins"] - consumo_tipico).abs()
    distancia_consumo = distancia_consumo.where(consumo_valido_tmp, np.inf)
    completitud_tmp = df.notna().sum(axis=1)
    return (
        pd.DataFrame({
            "_idx": df.index,
            "_login_valido": login_valido_tmp.astype(int),
            "_consumo_valido": consumo_valido_tmp.astype(int),
            "_distancia_consumo": distancia_consumo,
            "_login": login_tmp,
            "_complete": completitud_tmp
        })
        .sort_values(
            ["_login_valido", "_consumo_valido", "_distancia_consumo", "_login", "_complete", "_idx"],
            ascending=[False, False, True, False, False, True]
        )["_idx"]
    )

raw = pd.read_json(raw_path)
df = raw.copy()
filas_iniciales = len(df)
log = []

def registrar(paso, descripcion):
    """Guarda trazabilidad: paso aplicado, tamaño, nulos y retención."""
    log.append({
        "Paso": paso,
        "Descripción": descripcion,
        "Filas": len(df),
        "Nulos": int(df.isna().sum().sum()),
        "Retención (%)": round(len(df) / filas_iniciales * 100, 2)
    })

registrar("00", "Carga del dataset original en una copia de trabajo.")

# 1. Duplicados exactos: no aportan información nueva.
df = df.drop_duplicates().reset_index(drop=True)
registrar("01", "Eliminación de duplicados exactos sin modificar columnas.")

# 2. user_id repetidos: se conserva la fila con mayor calidad observable.
# Prioridad: fecha real, consumo mensual plausible, cercanía al consumo típico,
# fecha más reciente y completitud. Las columnas auxiliares son temporales.
orden = ordenar_usuarios_repetidos(df)
df = df.loc[orden].drop_duplicates(subset="user_id", keep="first").sort_values("user_id").reset_index(drop=True)
registrar("02", "Resolución de user_id repetidos priorizando fecha real, consumo mensual plausible, cercanía al consumo típico y completitud.")

# 3. Estandarización de categorías equivalentes.
mapa_plan = {"estándar":"Estándar", "estandar":"Estándar", "std":"Estándar", "standard":"Estándar",
             "básico":"Básico", "basico":"Básico", "basic":"Básico", "premium":"Premium", "premiun":"Premium"}
mapa_pais = {"argentina":"Argentina", "arg":"Argentina", "brasil":"Brasil", "brazil":"Brasil", "bra":"Brasil",
             "chile":"Chile", "chl":"Chile", "colombia":"Colombia", "col":"Colombia", "méxico":"México",
             "mexico":"México", "mex":"México", "perú":"Perú", "peru":"Perú", "per":"Perú",
             "uruguay":"Uruguay", "ury":"Uruguay"}
mapa_genero = {"acción":"Acción", "accion":"Acción", "action":"Acción", "comedia":"Comedia", "comedy":"Comedia",
               "crime":"Crimen", "crimen":"Crimen", "documental":"Documental", "documentary":"Documental",
               "doc":"Documental", "drama":"Drama", "romance":"Romance", "thriller":"Thriller", "thriler":"Thriller"}

df["subscription_plan"] = df["subscription_plan"].map(lambda x: mapa_plan.get(normalizar_texto(x), x))
df["country"] = df["country"].map(lambda x: mapa_pais.get(normalizar_texto(x), x))
df["favorite_genre"] = df["favorite_genre"].map(lambda x: mapa_genero.get(normalizar_texto(x), np.nan if pd.isna(x) else x))
registrar("03", "Estandarización de categorías en plan, país y género favorito.")

# 4. Valores imposibles: se convierten a nulo para imputarlos con criterios explícitos.
df.loc[(df["age"] < 13) | (df["age"] > 100), "age"] = np.nan
df.loc[df["monthly_watch_time_mins"] < 0, "monthly_watch_time_mins"] = np.nan
df.loc[df["customer_support_tickets"] < 0, "customer_support_tickets"] = np.nan
login = parsear_fechas_login(df["last_login_date"])
login.loc[login > pd.Timestamp("2026-06-28")] = pd.NaT
df["last_login_date"] = login
registrar("04", "Conversión de valores imposibles a nulos: edades fuera de 13-100, tiempos negativos, tickets negativos y fechas inválidas/futuras.")

# 5. Imputación: medianas para numéricas y modas para categóricas.
for col in ["age", "monthly_watch_time_mins"]:
    df[col] = df.groupby(["subscription_plan", "country"], observed=True)[col].transform(lambda s: s.fillna(s.median()))
    df[col] = df[col].fillna(df[col].median())

df["customer_support_tickets"] = df["customer_support_tickets"].fillna(df["customer_support_tickets"].median())
df["favorite_genre"] = df.groupby(["subscription_plan", "country"], observed=True)["favorite_genre"].transform(
    lambda s: s.fillna(s.mode().iloc[0] if not s.mode().empty else np.nan)
)
df["favorite_genre"] = df["favorite_genre"].fillna(df["favorite_genre"].mode().iloc[0])
df["last_login_date"] = df["last_login_date"].fillna(df["last_login_date"].dropna().median())
registrar("05", "Imputación justificada con medianas/modas segmentadas y fecha mediana global.")

# 6. Winsorización: se capean extremos superiores que distorsionan media, correlaciones y PCA.
cap_watch = df.loc[df["monthly_watch_time_mins"] <= 14400, "monthly_watch_time_mins"].quantile(0.99)
cap_tickets = df.loc[df["customer_support_tickets"] <= 30, "customer_support_tickets"].quantile(0.99)
df["monthly_watch_time_mins"] = df["monthly_watch_time_mins"].clip(upper=cap_watch)
df["customer_support_tickets"] = df["customer_support_tickets"].clip(upper=cap_tickets)
registrar("06", f"Winsorización superior: monthly_watch_time_mins cap={cap_watch:.1f}; customer_support_tickets cap={cap_tickets:.0f}.")

# 7. Tipos finales. Se respeta la estructura original: no se agregan columnas al dataset procesado.
df["age"] = df["age"].round().astype(int)
df["customer_support_tickets"] = df["customer_support_tickets"].round().astype(int)
df["monthly_watch_time_mins"] = df["monthly_watch_time_mins"].round(1)
df["last_login_date"] = pd.to_datetime(df["last_login_date"]).dt.strftime("%Y-%m-%d")
df = df[raw.columns]
registrar("07", "Normalización final de tipos y exportación del dataset procesado con las mismas columnas originales.")

df.to_csv(processed_path, index=False, encoding="utf-8")
pd.DataFrame(log).to_csv(log_path, index=False, encoding="utf-8")
pd.DataFrame(log)


In [ ]:
print("Columnas originales:", list(raw.columns))
print("Columnas procesadas:", list(df.columns))
print("¿Se agregaron columnas?", list(raw.columns) != list(df.columns))


In [ ]:
pd.read_csv(log_path)


## Decisiones principales

- Duplicados: se eliminaron duplicados exactos y se resolvieron `user_id` repetidos priorizando fecha real, consumo mensual plausible, cercanía al consumo típico y completitud.
- Categorías: se estandarizaron valores equivalentes sin crear columnas nuevas.
- Imposibles: valores fuera de rango razonable se trataron como nulos antes de imputar.
- Imputación: se usaron medianas por plan y país para numéricas, moda segmentada para género y fecha mediana para login.
- Winsorización: se aplicó en consumo mensual y tickets porque los extremos distorsionaban la escala y no representaban comportamiento normal.
